
# Group GEMM
This group gemm kernel launches a fixed number of CTA to compute a group
of gemms. The scheduling is static and we do it on device.


In [1]:
from typing import Optional
import torch

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()


def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"


def supports_tma():
    return is_cuda() and torch.cuda.get_device_capability()[0] >= 9


def num_sms():
    if is_cuda():
        return torch.cuda.get_device_properties("cuda").multi_processor_count
    return 148


@triton.autotune(
    configs=[
        triton.Config({
            'BLOCK_SIZE_M': 128,
            'BLOCK_SIZE_N': 128,
            'BLOCK_SIZE_K': 32,
            'NUM_SM': 84,
        }),
        triton.Config({
            'BLOCK_SIZE_M': 128,
            'BLOCK_SIZE_N': 128,
            'BLOCK_SIZE_K': 32,
            'NUM_SM': 128,
        }),
        triton.Config({
            'BLOCK_SIZE_M': 64,
            'BLOCK_SIZE_N': 64,
            'BLOCK_SIZE_K': 32,
            'NUM_SM': 84,
        }),
        triton.Config({
            'BLOCK_SIZE_M': 64,
            'BLOCK_SIZE_N': 64,
            'BLOCK_SIZE_K': 32,
            'NUM_SM': 128,
        }),
        triton.Config({
            'BLOCK_SIZE_M': 128,
            'BLOCK_SIZE_N': 128,
            'BLOCK_SIZE_K': 64,
            'NUM_SM': num_sms(),
        }),
        triton.Config({
            'BLOCK_SIZE_M': 64,
            'BLOCK_SIZE_N': 128,
            'BLOCK_SIZE_K': 64,
            'NUM_SM': num_sms(),
        }),
    ],
    key=['group_size'],
)
@triton.jit
def grouped_matmul_kernel(
    # device tensor of matrices pointers
    group_a_ptrs,
    group_b_ptrs,
    group_c_ptrs,
    # device tensor of gemm sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <M, N, K> of each gemm
    group_gemm_sizes,
    # device tensor of leading dimension sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <lda, ldb, ldc> of each gemm
    g_lds,
    # number of gemms
    group_size,
    # number of virtual SM
    NUM_SM: tl.constexpr,
    # tile sizes
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    tile_idx = tl.program_id(0)
    last_problem_end = 0
    for g in range(group_size):
        # get the gemm size of the current problem
        gm = tl.load(group_gemm_sizes + g * 3)
        gn = tl.load(group_gemm_sizes + g * 3 + 1)
        gk = tl.load(group_gemm_sizes + g * 3 + 2)
        num_m_tiles = tl.cdiv(gm, BLOCK_SIZE_M)
        num_n_tiles = tl.cdiv(gn, BLOCK_SIZE_N)
        num_tiles = num_m_tiles * num_n_tiles
        # iterate through the tiles in the current gemm problem
        while (tile_idx >= last_problem_end and tile_idx < last_problem_end + num_tiles):
            # pick up a tile from the current gemm problem
            k = gk
            lda = tl.load(g_lds + g * 3)
            ldb = tl.load(g_lds + g * 3 + 1)
            ldc = tl.load(g_lds + g * 3 + 2)
            a_ptr = tl.load(group_a_ptrs + g).to(tl.pointer_type(tl.float16))
            b_ptr = tl.load(group_b_ptrs + g).to(tl.pointer_type(tl.float16))
            c_ptr = tl.load(group_c_ptrs + g).to(tl.pointer_type(tl.float16))
            # figure out tile coordinates
            tile_idx_in_gemm = tile_idx - last_problem_end
            tile_m_idx = tile_idx_in_gemm // num_n_tiles
            tile_n_idx = tile_idx_in_gemm % num_n_tiles

            # do regular gemm here
            offs_am = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_bn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            offs_k = tl.arange(0, BLOCK_SIZE_K)
            a_ptrs = a_ptr + offs_am[:, None] * lda + offs_k[None, :]
            b_ptrs = b_ptr + offs_k[:, None] * ldb + offs_bn[None, :]
            accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
            for kk in range(0, tl.cdiv(k, BLOCK_SIZE_K)):
                # hint to Triton compiler to do proper loop pipelining
                tl.multiple_of(a_ptrs, [16, 16])
                tl.multiple_of(b_ptrs, [16, 16])
                # assume full tile for now
                a = tl.load(a_ptrs)
                b = tl.load(b_ptrs)
                accumulator += tl.dot(a, b)
                a_ptrs += BLOCK_SIZE_K
                b_ptrs += BLOCK_SIZE_K * ldb
            c = accumulator.to(tl.float16)

            offs_cm = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_cn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            c_ptrs = c_ptr + ldc * offs_cm[:, None] + offs_cn[None, :]

            # assumes full tile for now
            tl.store(c_ptrs, c)

            # go to the next tile by advancing NUM_SM
            tile_idx += NUM_SM

        # get ready to go to the next gemm problem
        last_problem_end = last_problem_end + num_tiles


def group_gemm_fn(group_A, group_B):
    assert len(group_A) == len(group_B)
    group_size = len(group_A)

    A_addrs = []
    B_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    group_C = []
    for i in range(group_size):
        A = group_A[i]
        B = group_B[i]
        assert A.shape[1] == B.shape[0]
        M, K = A.shape
        K, N = B.shape
        print(M, K, N)
        C = torch.empty((M, N), device=DEVICE, dtype=A.dtype)
        group_C.append(C)
        A_addrs.append(A.data_ptr())
        B_addrs.append(B.data_ptr())
        C_addrs.append(C.data_ptr())
        g_sizes += [M, N, K]
        g_lds += [A.stride(0), B.stride(0), C.stride(0)]

    # note these are device tensors
    d_a_ptrs = torch.tensor(A_addrs, device=DEVICE)
    d_b_ptrs = torch.tensor(B_addrs, device=DEVICE)
    d_c_ptrs = torch.tensor(C_addrs, device=DEVICE)
    d_g_sizes = torch.tensor(g_sizes, dtype=torch.int32, device=DEVICE)
    d_g_lds = torch.tensor(g_lds, dtype=torch.int32, device=DEVICE)
    # we use a fixed number of CTA, and it's auto-tunable
    grid = lambda META: (META['NUM_SM'], )
    grouped_matmul_kernel[grid](
        d_a_ptrs,
        d_b_ptrs,
        d_c_ptrs,
        d_g_sizes,
        d_g_lds,
        group_size,
    )

    return group_C

In [2]:
group_m = [16]
group_n = [2048]
group_k = [5192]
group_A = []
group_B = []
group_B_T = []
assert len(group_m) == len(group_n)
assert len(group_n) == len(group_k)
group_size = len(group_m)
for i in range(group_size):
    M = group_m[i]
    N = group_n[i]
    K = group_k[i]
    A = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    B = torch.rand((K, N), device=DEVICE, dtype=torch.float16)
    B_T = B.T.contiguous()
    group_A.append(A)
    group_B.append(B)
    group_B_T.append(B_T)

#tri_out = group_gemm_fn(group_A, group_B)
# ref_out = [torch.matmul(a, b) for a, b in zip(group_A, group_B)]
# for i in range(group_size):
#     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)


In [3]:
C = torch.empty((M, N), device=DEVICE, dtype=A.dtype)

In [4]:
C

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0', dtype=torch.float16)

In [5]:
C.stride()

(2048, 1)

In [6]:
A.shape

torch.Size([16, 5192])

In [7]:
tri_out = group_gemm_fn(group_A, group_B)

16 5192 2048


In [8]:
tri_out

[tensor([[1300., 1297., 1304.,  ..., 1283., 1297., 1289.],
         [1309., 1322., 1317.,  ..., 1309., 1310., 1303.],
         [1293., 1303., 1292.,  ..., 1289., 1294., 1291.],
         ...,
         [1293., 1311., 1297.,  ..., 1286., 1290., 1280.],
         [1279., 1270., 1275.,  ..., 1272., 1267., 1271.],
         [1284., 1303., 1286.,  ..., 1291., 1290., 1290.]], device='cuda:0',
        dtype=torch.float16)]

In [9]:
N = 10
K = 20
M = 64
num_experts = 128
num_activated_experts = min(M, num_experts) # take min of M and num_experts

In [10]:
#num_experts = min(M, 128) # M is batch_size, 128 is number of activaeted experts
dtype = torch.float16
num_activated_experts = min(M, num_experts)
group_a = []
group_b = []
# expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32)
for m in range(num_activated_experts):
    #group_a.append(torch.unsqueeze(a[m , :], 0))
    a = torch.randn((M, K), device=DEVICE, dtype=dtype)
    # b = torch.randn((num_experts, K, N), device=DEVICE, dtype= dtype)
    b = torch.randn((K, N), device=DEVICE, dtype= dtype)
    group_a.append(a)
    group_b.append(b)
    #group_b.append(b[expert_ids[m], :, :])

In [11]:
a.shape

torch.Size([64, 20])

## MoE Triton GEMM

In [13]:
def get_config():
    config_file_path = "config.json"
    if os.path.exists(config_file_path):
        with open(config_file_path) as f:
            return {int(key): val for key, val in json.load(f).items()}

@triton.jit
def matmul_kernel_moe(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        expert_ids_ptr, # NEW important!
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_be, stride_bk, stride_bn,  # add stride for expert outer 3D dimension
        stride_cm, stride_cn,
        top_k: tl.constexpr, # ADDDED! will select how many params
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
        # ACTIVATION: tl.constexpr  # don't need activation
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    # Program ID
    pid = tl.program_id(axis=0)
    # Number of program ids along the M axis
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    # Number of programs ids along the N axis
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    # Number of programs in group
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    # Id of the group this program is in
    group_id = pid // num_pid_in_group
    # Row-id of the first program in the group
    first_pid_m = group_id * GROUP_SIZE_M
    # If `num_pid_m` isn't divisible by `GROUP_SIZE_M`, the last group is smaller
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    # *Within groups*, programs are ordered in a column-major order
    # Row-id of the program in the *launch grid*
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    # Col-id of the program in the *launch grid*
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_be > 0) 
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    # NEW need to add expert dimension and pointer
    off_experts = tl.load(expert_ids_ptr + pid_m).to(tl.int64)
    # offs_token = tl.load(sorted_token_ids_ptr + offs_token_id)
    # NEW: integer division by top_k
    a_ptrs = a_ptr + (offs_am[:, None] // top_k * stride_am + offs_k[None, :] * stride_ak)
    # a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    # NEW
    b_ptrs = b_ptr + off_experts * stride_be + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
    #b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!
    # if ACTIVATION == "leaky_relu":
    #     accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    #c_ptrs = c_ptr + stride_cm * offs_token[:, None] + stride_cn * offs_cn[
     #   None, :] missing offs_token and token_mask
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [14]:
import os, json 

configs = get_config()
def matmul_moe(a, b, top_k_num=1, activation=""):
    # NEW
    # Check constraints.
    assert a.shape[1] == b.shape[2], "Incompatible dimensions" # UPDATED check
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape # new shape M is total expert we only want activated ones
    num_experts, N, K = b.shape # 3D shape NEW
    # Allocates output.
    c = torch.empty((M, top_k_num, N), device=a.device, dtype=torch.float16) # NEW 2nd dim to 2D
    # NEW
    config = configs[M]
    # 1D launch kernel where each block gets its own program.
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    # NEW: activate the experts
    activated_experts = min(M, num_experts) # take min of M and num_experts
    expert_ids = torch.arange(activated_experts, device=a.device, dtype=torch.int32).view(1, -1)
    matmul_kernel_moe[grid](
        a, b, c,  #
        M, N, K,  #
        expert_ids,
        a.stride(0), a.stride(1),  # important $(M,K)
        b.stride(0), b.stride(2), b.stride(1), # transposed b (top_k, K, N)
        c.stride(1), c.stride(2),  # CHECK c output is (M, top_k, N) top_k out of num_experts
        top_k = top_k_num,
        **config,
        # ACTIVATION=activation  #
    )
    return c

In [15]:
K = 5120
N = 2048 # divide by 8 for TP and multiply by 2 (original 8192)
use_fp_8 = False

In [16]:
niter = 1

In [17]:
M = 8
K = 10
N = 20

In [ ]:
#num_experts = min(M, 128) # M is batch_size, 128 is number of activaeted experts
dtype = torch.float16
num_experts = 128
M = 16
num_activated_experts = min(M, num_experts)
a = torch.randn((M, K), device=DEVICE, dtype=dtype)
b = torch.randn((num_experts, K, N), device=DEVICE, dtype= dtype)
group_a = []
group_b = []
activated_experts = min(M, num_experts) # take min of M and num_experts
expert_ids = torch.arange(activated_experts, device=a.device, dtype=torch.int32)
for m in range(num_activated_experts):
    group_a.append(torch.unsqueeze(a[m , :], 0))
    group_b.append(b[expert_ids[m], :, :])

RuntimeError: CUDA error: misaligned address
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 

In [22]:
for i in range(num_activated_experts):
    print(group_a[i].shape, group_b[i].shape)

torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])
torch.Size([64, 20]) torch.Size([20, 10])


In [23]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()

niter = 10
dtype=torch.float16
num_experts = 128
dtype_fp8 = torch.float8_e4m3fn
quantiles = [0.5, 0.2,  0.8]
use_fp_8 = True
res = []
for i in range(0, niter):
    M = 2 ** i
    #num_experts = min(M, 128) # M is batch_size, 128 is number of activaeted experits
    a = torch.randn((M, K), device=DEVICE, dtype=dtype)
    b = torch.randn((num_experts, N, K), device=DEVICE, dtype= dtype)
    num_activated_experts = min(M, 128)
    expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32)
    group_a = []
    group_b = []
    for m in range(num_activated_experts):
        group_a.append(torch.unsqueeze(a[m , :], 0)) # make it stay 2D row vector
        group_b.append(torch.swapaxes(b[expert_ids[m], :, :],0 , 1)) # need to transpose b
    if use_fp_8:
        a = a.to(dtype_fp8)
        # b = b.T
        b = b.to(dtype_fp8)

    if not use_fp_8:
        # cublas_ms = triton.testing.do_bench(lambda: torch.matmul(a, torch.swapaxes(b, 1, 2)), quantiles=quantiles)
        # b = torch.swapaxes(b, 1, 2)
        triton_ms = triton.testing.do_bench(lambda: matmul_moe(a, b), quantiles=quantiles)
        # print("M", M, "cublasms", cublas_ms)
    else:
        triton_ms = triton.testing.do_bench(lambda: matmul_moe(a, b), quantiles=quantiles)
        #triton_group_ms = triton.testing.do_bench(lambda: group_gemm_fn(group_a, group_b), quantiles=quantiles)
        
    res.append(triton_ms[1])
    print("M", M, "tritonms", triton_ms)

RuntimeError: CUDA error: misaligned address
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
